# 01 — What Is Agentic AI? (Code Examples)

This notebook demonstrates the difference between **traditional LLM usage** and **agentic AI** through progressive code examples.

**Prerequisites:** `pip install openai python-dotenv rich`  
**Setup:** Create a `.env` file with `OPENAI_API_KEY=your_key`

In [ ]:
# Setup
import os
from dotenv import load_dotenv
from openai import OpenAI
from rich import print as rprint
from rich.panel import Panel
from rich.table import Table

load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
print('✅ Setup complete')

## Example 1: Traditional LLM — Single Turn, No Memory, No Tools

This is **NOT** agentic. One input → one output. No persistence, no actions.

In [ ]:
# Traditional LLM Usage
def traditional_llm(question: str) -> str:
    """Single prompt → single response. No memory, no tools, no loops."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": question}
        ]
    )
    return response.choices[0].message.content

# Test it
question = "What is the current price of Bitcoin?"
answer = traditional_llm(question)

rprint(Panel(
    f"[bold]Question:[/bold] {question}\n\n[bold]Answer:[/bold] {answer}",
    title="🤖 Traditional LLM",
    border_style="red"
))

print("\n⚠️  Notice: The LLM CANNOT know current Bitcoin price — it has no tools.")
print("   It will either hallucinate, refuse, or give outdated training data.")

## Example 2: Agentic Approach — Same Question, With a Tool

Now the agent uses a **web search tool** to get the actual current price.

In [ ]:
import json

# ── Mock tool (replace with real Tavily/SerpAPI in production) ──
def web_search(query: str) -> str:
    """Mock web search tool — simulates real-time data retrieval."""
    # In reality: return requests.get('https://api.tavily.com/search', ...).json()
    mock_results = {
        "bitcoin price": "Bitcoin (BTC) current price: $67,342 USD (as of March 2025)",
        "capital of japan": "The capital of Japan is Tokyo.",
        "default": f"Search results for: {query} — [mock data returned]"
    }
    for key in mock_results:
        if key in query.lower():
            return mock_results[key]
    return mock_results["default"]

# ── Tool Schema (how we tell the LLM about available tools) ──
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for current, real-time information. Use when you need up-to-date data not in your training knowledge.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query to find information about"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

def agentic_llm(question: str) -> str:
    """Agentic LLM with tool use — can access real-world information."""
    messages = [
        {"role": "system", "content": "You are a helpful AI agent. Use the web_search tool to get current information when needed."},
        {"role": "user", "content": question}
    ]
    
    print(f"Step 1 → Sending to LLM: '{question}'")
    
    # First LLM call: decide whether to use a tool
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
        tool_choice="auto"
    )
    
    msg = response.choices[0].message
    
    # Did the LLM decide to use a tool?
    if msg.tool_calls:
        tool_call = msg.tool_calls[0]
        tool_name = tool_call.function.name
        tool_args = json.loads(tool_call.function.arguments)
        
        print(f"Step 2 → LLM decided to use tool: {tool_name}")
        print(f"         Arguments: {tool_args}")
        
        # Execute the tool
        tool_result = web_search(tool_args["query"])
        print(f"Step 3 → Tool result: {tool_result}")
        
        # Add tool call and result to message history
        messages.append(msg)  # LLM's tool call decision
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": tool_result
        })
        
        # Second LLM call: generate final answer using tool result
        print("Step 4 → Sending tool result back to LLM for final answer...")
        final_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages
        )
        return final_response.choices[0].message.content
    else:
        # LLM answered directly without tools
        return msg.content

# Test the agentic version
print("=" * 60)
result = agentic_llm("What is the current price of Bitcoin?")
print("=" * 60)
rprint(Panel(
    result,
    title="🤖 Agentic LLM Answer",
    border_style="green"
))

## Example 3: The Core Difference — Side by Side Comparison

In [ ]:
# Visual comparison of Traditional vs Agentic AI
table = Table(title="Traditional LLM vs Agentic AI", show_header=True)
table.add_column("Dimension", style="bold cyan", width=25)
table.add_column("Traditional LLM", style="red", width=30)
table.add_column("Agentic AI", style="green", width=30)

rows = [
    ("Interaction style", "Single prompt → response", "Multi-step goal-pursuing loop"),
    ("Memory", "None (stateless per call)", "Persistent short + long-term"),
    ("Tools", "None", "Web, code, APIs, databases"),
    ("Decision making", "Reactive (responds to input)", "Proactive (self-directed)"),
    ("Error handling", "None", "Self-corrects, re-plans"),
    ("Context", "One turn", "Spans sessions, resumes tasks"),
    ("Example", "Q&A chatbot", "Devin, Perplexity, Cursor"),
]

for row in rows:
    table.add_row(*row)

rprint(table)

## Example 4: Three Core Properties of an Agent

Demonstrating **Autonomy**, **Goal-directedness**, and **Proactiveness** in code.

In [ ]:
# Demonstrating AUTONOMY: agent decides its OWN next steps
def demonstrate_autonomy():
    """Autonomy: LLM decides what to do next without being told each step."""
    
    goal = "Find out who won the most recent FIFA World Cup and when it was held."
    
    messages = [
        {
            "role": "system",
            "content": (
                "You are an autonomous agent. When given a goal, decide yourself whether "
                "you need to use tools or can answer directly. Think step by step."
            )
        },
        {"role": "user", "content": f"Goal: {goal}"}
    ]
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
        tool_choice="auto"  # LLM AUTONOMOUSLY decides to use tool or not
    )
    
    msg = response.choices[0].message
    decision = "Used web_search tool" if msg.tool_calls else "Answered from training knowledge"
    
    print(f"Goal: {goal}")
    print(f"Agent's autonomous decision: {decision}")
    if msg.tool_calls:
        args = json.loads(msg.tool_calls[0].function.arguments)
        print(f"Agent chose query: '{args['query']}'")
    print("\n→ KEY INSIGHT: We didn't tell the agent WHETHER to search — it decided itself.")

demonstrate_autonomy()

## Example 5: What Makes Code Agentic — The Loop

The key structural difference: **a loop with a termination condition**.

In [ ]:
def minimal_agent_loop(goal: str, max_steps: int = 5) -> str:
    """
    The most minimal agentic loop possible:
    - Takes a goal
    - Loops: think → act → observe
    - Terminates when done or max_steps reached
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are an agent. Use web_search to find information. "
                "When you have enough information to fully answer the goal, "
                "respond with your final answer WITHOUT using any tools."
            )
        },
        {"role": "user", "content": f"Goal: {goal}"}
    ]
    
    for step in range(1, max_steps + 1):
        print(f"\n{'─'*50}")
        print(f"STEP {step}")
        print(f"{'─'*50}")
        
        # ── REASONING ── Ask LLM what to do next
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=TOOLS,
            tool_choice="auto"
        )
        
        msg = response.choices[0].message
        
        # ── TERMINATION CHECK ── No tool call = agent is done
        if not msg.tool_calls:
            print(f"✅ Agent finished in {step} step(s)")
            print(f"Final Answer: {msg.content}")
            return msg.content
        
        # ── ACTION ── Execute the tool the agent chose
        tool_call = msg.tool_calls[0]
        tool_args = json.loads(tool_call.function.arguments)
        print(f"Thought: Agent decides to search")
        print(f"Action: web_search('{tool_args['query']}')")        
        
        # ── OBSERVATION ── Get tool result
        observation = web_search(tool_args["query"])
        print(f"Observation: {observation}")
        
        # ── MEMORY (STM) ── Add to message history
        messages.append(msg)
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": observation
        })
    
    return "Max steps reached without completing goal."

# Run the minimal agent loop
result = minimal_agent_loop(
    goal="What is the current price of Bitcoin and the capital of Japan?",
    max_steps=5
)

## 🧠 Key Takeaways

1. **Traditional LLM**: prompt → response (no loop, no tools, no memory)
2. **Agentic AI** = LLM + Tools + Memory + Loop + Termination Condition
3. The **loop** is the fundamental structural difference
4. **Autonomy** means the LLM decides its own next steps — not us
5. Every framework (LangChain, CrewAI, Agno) wraps this same core loop